# Finding emission-line candidates in the Vela X-1 RGS spectrum

This worked example uses **RGS1, first order, observation 0841890201**, with a search interval of **0.62–0.92 keV**. It illustrates loading a spectrum, exploring rebinning, inspecting an empirical baseline, detecting candidates, evaluating them with synthetic spectra, and fitting and identifying selected features.


In [ ]:
# Import the public BLiSS API and the utilities used throughout this notebook.
from bliss import *
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from time import perf_counter

from bliss.line_search.blind_line_search import (
    BlindLineSearchConfig,)

### 1.1 Define the configuration

`BlindLineSearchConfig` holds the search settings. Commented-out arguments use the library defaults; they do not inherit values from earlier cells. Here the active overrides include a 0.1 keV energy margin, a maximum Gaussian sigma of 0.005 keV (5 eV), and Poisson simulations.

**Execution dependency:** `en1` and `en2` are first assigned later in the saved notebook. For a fresh session, define `en1, en2 = 0.62, 0.92` before this cell.

Creating `pipeline` does not execute a search. The later calls to `find_candidate_lines` construct their own pipeline. Only the evaluated search explicitly receives this `config`; keyword arguments in that call can override it.


In [ ]:
en1 = 0.5
en2 = 2.2
config = BlindLineSearchConfig(en1=en1, en2=en2, energy_pad=0.1, max_sigma_line=0.005, noise_model='poisson')
pipeline = BlindLineSearchPipeline(config=config)


## 2. Locate and load the RGS files and load the spectrum

| File | Role |
|---|---|
| `vela_RGS1_order1_grp.fits` | Source spectrum with grouping flags previously written by SAS |
| `P0841890201R1S004RSPMAT1001.FIT` | Corresponding RGS response |
| `P0841890201R1S004BGSPEC1001.FIT` | Background spectrum |


In [ ]:
# Load a background-subtracted spectrum; SAS GROUPING flags are not applied by this loader.
rgs_path = Path("./0841890201/odf/")  # Directory containing the FITS files

pha = rgs_path / "vela_RGS1_order1_grp.fits"
rmf = rgs_path / "P0841890201R1S004RSPMAT1001.FIT"
bkg = rgs_path / "P0841890201R1S004BGSPEC1001.FIT"

spectrum = prepare_spectrum(
    pha_path=pha,
    rmf_path=rmf,
    background_path=bkg,
)

## 3. Inspect the native spectrum and compare rebinning choices

The plotting helper restricts the plotted arrays to the requested energy interval. The printed median bin width is computed over the entire loaded spectrum, not just the displayed interval.

### 3.1 Native channels

`rebin_method="none"` preserves the retained input channels. **Unlike Section 2, this call does not pass a background file**, so `native` contains the source-region spectrum without background subtraction. It is a separate object from `spectrum`.


In [ ]:
def plot_spectrum(spectra, labels, xlim, title=None):
    plt.figure(figsize=(9, 4.5))
    for i, (s, lab) in enumerate(zip(spectra, labels)):
        m = (s.energy >= xlim[0]) & (s.energy <= xlim[1])
        plt.errorbar(s.energy[m], s.values[m], yerr=s.uncertainties[m],
                     fmt=".", ms=3 if i == 0 else 4, alpha=0.3 if i == 0 else 0.9, label=lab)
    plt.xlabel("Energy (keV)")
    plt.ylabel("counts s$^{-1}$ keV$^{-1}$")
    if title:
        plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


native = prepare_spectrum(pha, rmf, rebin_method="none")
print(f"{len(native.energy)} channels, median width {1e3*np.median(native.bin_width):.2f} eV, "
      f"exposure {native.native.exposure:.0f} s")

plot_spectrum([native], ["RGS Vela X-1"], xlim=(0.62, 0.92))


### 3.2 Group by signal-to-noise ratio

`rebin_method="snr", rebin_scale=3.0` accumulates adjacent input bins until their combined signal-to-noise ratio reaches 3. This can produce wider bins in weak parts of the spectrum; a trailing group can fall below the requested threshold.


In [ ]:
# S/N-based grouping of the source-region spectrum (no background passed here).
spectrum_snr = prepare_spectrum(
    pha,
    rmf,
    rebin_method="snr",
    rebin_scale=3.0,
)

print(
    f"{len(spectrum_snr.energy)} bins, "
    f"median width {1e3 * np.median(spectrum_snr.bin_width):.2f} eV, "
    f"exposure {spectrum_snr.native.exposure:.0f} s"
)

plot_spectrum(
    [spectrum_snr],
    ["RGS Vela X-1 — S/N ≥ 3"],
    xlim=(0.62, 0.92),
)

### 3.3 Group a fixed number of channels

`rebin_method="bins", rebin_scale=3` combines groups of three retained input bins, sums their values, and propagates errors in quadrature. The implementation merges any incomplete trailing group into the preceding group. This is a channel-count criterion, not a constant energy width.

Here `background_path=bkg` is supplied, so `spectrum_bins` is background-subtracted. To compare grouping methods alone, all examples would need the same treatment of the background.


In [ ]:
# Combine three input bins per group after background subtraction.
spectrum_bins = prepare_spectrum(
    pha_path=pha,
    rmf_path=rmf,
    background_path=bkg,
    rebin_method="bins",
    rebin_scale=3,
)

print(
    f"{len(spectrum_bins.energy)} bins, "
    f"median width {1e3 * np.median(spectrum_bins.bin_width):.2f} eV, "
    f"exposure {spectrum_bins.native.exposure:.0f} s"
)

plot_spectrum(
    [spectrum_bins],
    ["RGS Vela X-1 — agrupado de 3 en 3"],
    xlim=(0.62, 0.92),
)

### 3.4 Repeated fixed-channel grouping cell

For a separate fixed-energy-width experiment, BLiSS accepts `rebin_method="resolution", rebin_scale=0.002` (2 eV). That method uses fixed energy intervals; it does not automatically follow the RGS instrumental resolution.


In [ ]:
# This repeats the preceding fixed-channel example; it is not resolution-based grouping.
spectrum_bins = prepare_spectrum(
    pha_path=pha,
    rmf_path=rmf,
    background_path=bkg,
    rebin_method="bins",
    rebin_scale=3,
)

print(
    f"{len(spectrum_bins.energy)} bins, "
    f"median width {1e3 * np.median(spectrum_bins.bin_width):.2f} eV, "
    f"exposure {spectrum_bins.native.exposure:.0f} s"
)

plot_spectrum(
    [spectrum_bins],
    ["RGS Vela X-1 — agrupado de 3 en 3"],
    xlim=(0.62, 0.92),
)

## 4. Inspect the empirical baseline

BLiSS estimates the baseline using a running median in energy. This provides an empirical description of the observed spectral shape rather than a physical continuum model. The method exploits the difference in scale between the smoothly varying continuum and narrow emission-line features. A window that is too narrow may absorb part of the line structure into the baseline, whereas a window that is too broad may fail to follow relatively rapid changes in the underlying continuum. If different spectral regions require different smoothing scales, an array of window sizes can also be provided.


In [ ]:
baseline_window = 0.4 #Default

baseline_bins, info_bins = base_calculator(
    spectrum_bins.energy,
    spectrum_bins.values,
    baseline_window=baseline_window,
    return_info=True)


fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(
    spectrum_bins.energy,
    spectrum_bins.values,
    yerr = spectrum_bins.uncertainties,
    fmt=".",
    color="black",
    linewidth=0.8,
   
    label="RGS Vela X-1", 
)

ax.plot(
    spectrum_bins.energy,
    baseline_bins,
    color="tab:orange",
    linewidth=2,
    label="Baseline",
)

ax.set(
    xlim=(0.62, 0.92),
    xlabel="Energía (keV)",
    ylabel=r"Cuentas s$^{-1}$ keV$^{-1}$",
)

plt.ylim(-0.05,0.25)
ax.legend()
fig.tight_layout()
plt.show()

### 4.2 Compare a smaller requested window: 0.01 keV

This cell recomputes `baseline_bins` with a 10 eV window. It overwrites the previous baseline but does not update `config.baseline_window`.


In [ ]:
# Compare a smaller baseline window on the same spectrum.
# TODO: use info_bins in the print and baseline_bins in the plotted curve.
baseline_window = 0.01


baseline_bins, info_bins = base_calculator(
    spectrum_bins.energy,
    spectrum_bins.values,
    baseline_window=baseline_window,
    return_info=True,
)


fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(
    spectrum_bins.energy,
    spectrum_bins.values,
    yerr = spectrum_bins.uncertainties,
        fmt=".",
    color="black",
    linewidth=0.8,
    label="RGS Vela X-1", 
)

ax.plot(
    spectrum_bins.energy,
    baseline_bins,
    color="tab:orange",
    linewidth=2,
    label="Baseline",
)

ax.set(
    xlim=(0.62, 0.92),
    xlabel="Energía (keV)",
    ylabel=r"Cuentas s$^{-1}$ keV$^{-1}$",
)

plt.ylim(-0.05,0.25)
ax.legend()
fig.tight_layout()
plt.show()

## 5. Run a quick candidate search without simulations

Setting `num_synthetic_simulations=0` detects and locally fits candidates without assigning simulation-based significance. This is useful for an initial inspection; the resulting candidates are not confirmed detections.

This call uses **two-bin grouping**, omits the background, and does **not** receive `config`. It therefore uses the default baseline and line-width settings rather than the custom configuration above. The timer measures the entire function call, including loading and preparation.


In [ ]:
# Quick search: no simulations, no background, and no custom config.
# The two-bin grouping differs from the evaluated search below.
en1, en2 = 0.62, 0.92

inicio = perf_counter()

candidate_lines_not_evaluated = find_candidate_lines(
    pha=pha,
    rmf=rmf,
    rebin_method="bins",
    rebin_scale=2,
    en1=en1,
    en2=en2,
    num_synthetic_simulations=0,
    synthetic_seed=1,
)

tiempo = perf_counter() - inicio

print(f"Tiempo total: {tiempo:.2f} s ({tiempo / 60:.2f} min)")

### 5.1 Overlay candidate energies

The vertical markers show the centers returned by the quick search.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(
    spectrum_bins.energy,
    spectrum_bins.values,
    yerr = spectrum_bins.uncertainties,
        fmt=".",
    color="black",
    linewidth=0.8,
    label="RGS Vela X-1", 
)

ax.set(
    xlim=(0.62, 0.92),
    xlabel="Energía (keV)",
    ylabel=r"Cuentas s$^{-1}$ keV$^{-1}$",
)

for i in range(len(candidate_lines_not_evaluated )):
    plt.vlines(x=candidate_lines_not_evaluated .center[i], ymin=-1, ymax=2)



plt.ylim(-0.05,0.35)
ax.legend()
fig.tight_layout()
plt.show()

## 6. Evaluate candidates with 100 synthetic spectra

The evaluated search uses three-bin grouping and a fixed random seed. Poisson null spectra provide a reference for the five `p_global_*` statistics.

With 100 simulations, the smallest attainable Monte Carlo p-value is **1 / (100 + 1) ≈ 0.0099**. Each global p-value compares one statistic with the maximum found across the search interval in each null realization.


In [ ]:
en1, en2 = (0.62, 0.92)
inicio = perf_counter()
candidate_lines_evaluated = find_candidate_lines(pha=pha, rmf=rmf, rebin_method='bins', rebin_scale=2, en1=en1, en2=en2, num_synthetic_simulations=100, synthetic_seed=1, config=config)
tiempo = perf_counter() - inicio
print(f'Tiempo total: {tiempo:.2f} s ({tiempo / 60:.2f} min)')


### 6.1 Inspect the evaluated candidate catalogue

Review fitted centers and widths, their uncertainties, areas, equivalent widths, global p-values.


In [ ]:
candidate_lines_evaluated

## 7. Select candidates and perform a global fit

The example jointly fits all candidate line profiles. Apply your chosen scientific selection before fitting when needed.


In [ ]:
candidate_lines_evaluated_filtered = candidate_lines_evaluated.copy()
lines_res_global = fit_global(candidate_lines_evaluated_filtered, spectrum_snr, baseline_window=0.025, energy_min=0.62, energy_max=0.92)


## 8. Assign tentative atomic identifications

`add_most_probable_ion` compares each fitted center with the bundled atomic line table, using a matching tolerance corresponding to 600 km s⁻¹.


In [ ]:
global_lines = add_most_probable_ion(lines_res_global, v_doppler_kms=600)
global_lines